<a href="https://colab.research.google.com/github/Loopinlogix/Market_Analysis_Project-2/blob/main/Stock_Market_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Stock Market Analysis Project 2


In [2]:

#Github

#Github
!apt-get install -y git
!git config --global user.email "crystal_macneil@hotmail.com"
!git config --global user.name "Crystal MacNeil"

!git clone https://github.com/Loopinlogix/Market_Analysis_Project-2.git
%cd Market_Analysis_Project-2
!ls


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Cloning into 'Market_Analysis_Project-2'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/Market_Analysis_Project-2
README.md


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 60)
print("STEP 1: DATA COLLECTION")
print("=" * 60)

# Load datasets
stocks = pd.read_csv('historical_stocks.csv')
prices = pd.read_csv('historical_stock_prices.csv')

# Standardize column names
stocks.columns = stocks.columns.str.strip().str.lower()
prices.columns = prices.columns.str.strip().str.lower()

print("Prices head:")
print(prices.head())
print("\nStocks head:")
print(stocks.head())
print("\nPrices info:")
print(prices.info())

# Merge datasets on ticker
df = pd.merge(prices, stocks, on='ticker', how='left')

print("=" * 60)
print("STEP 2: ADVANCED CLEANING")
print("=" * 60)

# Fill missing categorical values
categorical_cols = ['exchange', 'name', 'sector', 'industry']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

print("\nMissing values after categorical fill:")
print(df.isnull().sum())

display(df.head())

# Convert date column and drop critical missing values
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df.dropna(subset=['date', 'high', 'volume'], inplace=True)

print("\nMissing values after date conversion and dropping NaNs:")
print(df.isnull().sum())

print(f"\nData type of 'date': {df['date'].dtype}")

# Sort by date and ticker
df.sort_values(by=['date', 'ticker'], inplace=True)
print("\nSorted DataFrame head:")
display(df.head())

print("=" * 60)
print("STEP 3: OUTLIER HANDLING")
print("=" * 60)

print("Descriptive statistics BEFORE outlier clipping:")
print(df[['close', 'volume']].describe())

for col in ['close', 'volume']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

    print(f"Outliers capped for: {col}")

print("\nDescriptive statistics AFTER outlier clipping:")
print(df[['close', 'volume']].describe())

print("=" * 60)
print("STEP 4: DUPLICATE CHECK")
print("=" * 60)

initial_rows = df.shape[0]
duplicate_count = df.duplicated(subset=['ticker', 'date']).sum()

print(f"Initial rows: {initial_rows}")
print(f"Duplicate rows found (ticker + date): {duplicate_count}")

df.drop_duplicates(subset=['ticker', 'date'], inplace=True)

removed = initial_rows - df.shape[0]
print(f"Duplicate rows removed: {removed}")
print(f"Final DataFrame shape: {df.shape}")

STEP 1: DATA COLLECTION
Prices head:
  ticker   open  close  adj_close    low   high     volume        date
0    AHH  11.50  11.58   8.493155  11.25  11.68  4633900.0  2013-05-08
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800.0  2013-05-09
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100.0  2013-05-10
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400.0  2013-05-13
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100.0  2013-05-14

Stocks head:
  ticker exchange                                    name             sector  \
0    PIH   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
1  PIHPP   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
2   TURN   NASDAQ                180 DEGREE CAPITAL CORP.            FINANCE   
3   FLWS   NASDAQ                 1-800 FLOWERS.COM, INC.  CONSUMER SERVICES   
4   FCCY   NASDAQ           1ST CONSTITUTION BANCORP (NJ)            FINANCE   

                     industry  
0  PROPERT

,ticker,open,close,adj_close,low,high,volume,date,exchange,name,sector,industry
0,AHH,11.50,11.58,8.493155,11.25,11.68,4633900.0,2013-05-08,NYSE,"ARMADA HOFFLER PROPERTIES, INC.",FINANCE,REAL ESTATE
1,AHH,11.66,11.55,8.471151,11.50,11.66,275800.0,2013-05-09,NYSE,"ARMADA HOFFLER PROPERTIES, INC.",FINANCE,REAL ESTATE
2,AHH,11.55,11.60,8.507822,11.50,11.60,277100.0,2013-05-10,NYSE,"ARMADA HOFFLER PROPERTIES, INC.",FINANCE,REAL ESTATE
3,AHH,11.63,11.65,8.544494,11.55,11.65,147400.0,2013-05-13,NYSE,"ARMADA HOFFLER PROPERTIES, INC.",FINANCE,REAL ESTATE
4,AHH,11.60,11.53,8.456484,11.50,11.60,184100.0,2013-05-14,NYSE,"ARMADA HOFFLER PROPERTIES, INC.",FINANCE,REAL ESTATE



Missing values after date conversion and dropping NaNs:
ticker       0
open         0
close        0
adj_close    0
low          0
high         0
volume       0
date         0
exchange     0
name         0
sector       0
industry     0
dtype: int64

Data type of 'date': datetime64[ns]

Sorted DataFrame head:


,ticker,open,close,adj_close,low,high,volume,date,exchange,name,sector,industry
2515733,CNP,11.099500,11.169750,0.107916,10.994125,11.204875,24400.0,1970-01-02,NYSE,"CENTERPOINT ENERGY, INC.",PUBLIC UTILITIES,ELECTRIC UTILITIES: CENTRAL
1669485,MMM,6.851562,6.851562,0.438697,6.843750,6.890625,72000.0,1970-01-02,NYSE,3M COMPANY,HEALTH CARE,MEDICAL/DENTAL INSTRUMENTS
3754576,MRK,1.565972,1.545139,0.268152,1.541667,1.565972,475200.0,1970-01-02,NYSE,"MERCK & COMPANY, INC.",HEALTH CARE,MAJOR PHARMACEUTICALS
3892482,MRO,5.633611,5.757882,0.203525,5.633611,5.757882,105900.0,1970-01-02,NYSE,MARATHON OIL CORPORATION,ENERGY,OIL & GAS PRODUCTION
2515734,CNP,11.169750,11.345375,0.109612,11.169750,11.345375,13700.0,1970-01-05,NYSE,"CENTERPOINT ENERGY, INC.",PUBLIC UTILITIES,ELECTRIC UTILITIES: CENTRAL


STEP 3: OUTLIER HANDLING
Descriptive statistics BEFORE outlier clipping:
              close        volume
count  4.128489e+06  4.128489e+06
mean   1.408629e+02  1.725287e+06
std    5.345625e+03  2.772490e+07
min    1.000000e-03  1.000000e+00
25%    7.103720e+00  2.010000e+04
50%    1.512500e+01  1.159000e+05
75%    2.926000e+01  5.546000e+05
max    1.347500e+06  4.483504e+09
Outliers capped for: close
Outliers capped for: volume

Descriptive statistics AFTER outlier clipping:
              close        volume
count  4.128489e+06  4.128489e+06
mean   2.113193e+01  3.702101e+05
std    1.823378e+01  4.824238e+05
min    1.000000e-03  1.000000e+00
25%    7.103720e+00  2.010000e+04
50%    1.512500e+01  1.159000e+05
75%    2.926000e+01  5.546000e+05
max    6.249442e+01  1.356350e+06
STEP 4: DUPLICATE CHECK
Initial rows: 4128489
Duplicate rows found (ticker + date): 0
Duplicate rows removed: 0
Final DataFrame shape: (4128489, 12)
